# Exercise 1: FFT eines Dreieckssignals

Computational Physics · Universität Wien · 14.10.2026

Angabe: [`exercise.pdf`](exercise.pdf) · Theorie: [`THEORIE.md`](THEORIE.md)

**Inhalt**

0. Setup und das Signal
1. DFT
   a) naive DFT und inverse DFT, Spektrum und Zerlegung
   b) Vergleich mit `numpy.fft`
   c) Laufzeit DFT gegen FFT
   d) Vergleich mit dem analytischen Ergebnis
2. Eigenschaften der DFT
   a) Symmetrie · b) Verschiebungssatz · c) Rücktransformation und Parseval · d) Cooley-Tukey-Rekursion · e) rekursive FFT
3. Andere Signale
   a) reell, imaginär, komplex · b) Abfall und Glattheit · c) Signal passt nicht ins Fenster
4. Zusammenfassung

**Vorgehen:** Zu jeder Implementierung gibt es eine Test-Zelle mit `assert`. Die Tests werden zuerst geschrieben, dann der Code.

## 0 Setup und das Signal

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

### Das Dreieckssignal

Aus der Abbildung der Angabe:

- Amplitude 1, Periode $P = 2$ ms, Grundfrequenz $f_0 = 1/P = 500$ Hz
- $a(0) = 1$, Minimum $-1$ bei $t = 1$ ms
- Fenster $T = 8$ ms (4 Perioden)

Abtastpunkte $t_j = j\,\Delta t$, $j = 0, \dots, N-1$, $\Delta t = T/N$.
Der Endpunkt $t = T$ gehört **nicht** dazu (die DFT setzt das Signal periodisch fort).

### Notation

| Größe | Bedeutung | Index |
|---|---|---|
| $a_j = a(t_j)$ | Abtastwerte des Signals (keine Koeffizienten, sondern Messwerte) | Zeitpunkt $j$ |
| $\hat a_k$ | DFT: wie stark die Frequenz $f_k = k/T$ vorkommt; Betrag = Amplitude, Winkel = Phase | Frequenz $k$ |
| $\tilde a(\omega)$ | kontinuierliche Fourier-Transformation $\int a(t)\,e^{-i\omega t}\,dt$, $\omega = 2\pi f$ | kontinuierlich |

Umrechnung: $\hat a_k / N \approx c_k$ (Fourier-Reihen-Koeffizient), $\Delta t\,\hat a_k \approx \tilde a(\omega_k)$ mit $\omega_k = 2\pi k/T$.
Die DFT ist ein Basiswechsel von der Zeitbasis in die Frequenzbasis.

In [ ]:
# Parameter (Zeit in ms, Frequenz damit in kHz)
T = 8.0   # Fensterlänge
P = 2.0   # Periode des Dreiecks
N = 64    # Anzahl Abtastpunkte (Zweierpotenz, wegen der Radix-2-FFT in 2e)

### Abtastpunkte

$t_j = j\,\Delta t$ für $j = 0, \dots, N-1$. Bewusst `np.arange(n) * window / n` und nicht
`np.linspace(0, window, n)`: `linspace` würde den Endpunkt $t = T$ mitnehmen, und der ist
für die periodisch fortgesetzte DFT derselbe Punkt wie $t = 0$.

In [ ]:
def sample_times(n, window=T):
    """The n sampling times t_j = j * window / n, j = 0 ... n-1 (without the end point)."""
    return np.arange(n) * window / n

### Abtastschritt, Frequenzauflösung und Nyquist-Frequenz

Aus $T$ und $N$ folgen die Kenngrößen der Abtastung:

| Größe | Formel | Bedeutung |
|---|---|---|
| Abtastschritt | $\Delta t = T/N$ | Abstand der Abtastpunkte |
| Abtastrate | $f_s = 1/\Delta t$ | Abtastpunkte pro Zeiteinheit |
| Frequenzauflösung | $\Delta f = 1/T$ | Abstand der DFT-Frequenzen $f_k = k\,\Delta f$ |
| Nyquist-Frequenz | $f_\text{Nyquist} = f_s/2 = \dfrac{1}{2\Delta t} = \dfrac{N}{2T}$ | höchste darstellbare Frequenz ($k = N/2$) |

Eine Schwingung braucht mindestens zwei Abtastpunkte pro Periode, damit man sie erkennt. Deshalb liegt die
Grenze bei der halben Abtastrate. Frequenzen darüber werden an $f_\text{Nyquist}$ in den Bereich
$0 \dots f_\text{Nyquist}$ zurückgespiegelt (Aliasing, `THEORIE.md` Abschnitt 4 und Aufgabe 1d).

Zeit in ms → Frequenzen in kHz.

In [ ]:
def nyquist_frequency(n, window=T):
    """Highest frequency the DFT of n samples in [0, window) can represent: 1 / (2 dt) = n / (2 window)."""
    dt = window / n
    return 1 / (2 * dt)


# tests: 8 samples per ms -> 4 kHz ; equals the largest |frequency| of np.fft.fftfreq (index k = N/2)
assert np.isclose(nyquist_frequency(64, window=8.0), 4.0)
assert np.isclose(nyquist_frequency(N), np.max(np.abs(np.fft.fftfreq(N, T / N))))

dt = T / N
f_nyquist = nyquist_frequency(N)
print(f"N = {N}, T = {T:g} ms")
print(f"  Abtastschritt       Δt          = {dt:g} ms")
print(f"  Abtastrate          f_s         = {1 / dt:g} kHz")
print(f"  Frequenzauflösung   Δf          = {1 / T:g} kHz")
print(f"  Nyquist-Frequenz    f_Nyquist   = {f_nyquist:g} kHz   (ω_Nyquist = {2 * np.pi * f_nyquist:.2f} rad/ms)")
harmonics_below = np.arange(1, int(f_nyquist * P) + 1, 2)
print(f"  Oberschwingungen des Dreiecks unterhalb Nyquist: {[f'{n / P:g} kHz' for n in harmonics_below]}")

### Funktionswerte des Dreiecks

Zwei unabhängige Implementierungen. Weil beide auf verschiedenen Wegen rechnen, ist ihre
Übereinstimmung selbst schon ein Test.

**Tests zuerst.** Die Prüfungen stehen in einer Funktion, die beide Varianten durchlaufen:
- bekannte Punkte: $a(0) = 1$, $a(0.5) = 0$, $a(1) = -1$, $a(1.5) = 0$
- Periodizität: $a(t + P) = a(t)$, auch für negative $t$
- Wertebereich $[-1, 1]$
- gerade Funktion: $a(-t) = a(t)$ (wichtig für Aufgabe 2b)
- andere Perioden: $a(P/2) = -1$ auch für $P \ne 2$

In [ ]:
def check_triangle(f):
    """Tests for an implementation f(t, period) of the triangle signal."""
    # known points
    t = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 7.5])
    assert np.allclose(f(t), [1, 0, -1, 0, 1, 0])

    # periodic, also for negative t
    t = np.linspace(-5, 13, 1001)
    assert np.allclose(f(t + P), f(t))

    # range [-1, 1]
    assert f(t).min() >= -1 - 1e-12 and f(t).max() <= 1 + 1e-12

    # even function: a(-t) = a(t)
    assert np.allclose(f(-t), f(t))

    # other periods: maximum at 0, minimum at period/2
    for period in (0.5, 3.0):
        assert np.allclose(f(np.array([0.0, period / 2, period]), period), [1, -1, 1])


def check_sample_times():
    t = sample_times(8, window=8.0)
    assert np.allclose(t, [0, 1, 2, 3, 4, 5, 6, 7])  # end point 8 is not included
    assert len(sample_times(N)) == N
    assert np.isclose(sample_times(N)[1], T / N)     # dt = T / N

#### Variante 1: Betragsfunktion mit Modulo (Idee: Emil)

**Idee:** Die Betragsfunktion $|x|$ ist schon ein „halbes Dreieck“ mit Knick bei 0.
Mit $x = t \bmod 2$ wird sie periodisch. Damit der Knick in der Mitte der Periode liegt,
wird sie um 1 nach rechts verschoben, und um 1 nach unten, damit das Signal um 0 schwingt.

**Korrektur bei der Umsetzung:** $|x - 1| - 1$ hat die richtige Form und Periode, läuft aber
nur von $-1$ bis $0$, weil $|x - 1|$ auf $[0, 2)$ nur Werte von 0 bis 1 annimmt.
Für die Amplitude 1 muss zusätzlich um den Faktor 2 gestreckt werden:

$$a(t) = 2\,|x - 1| - 1, \qquad x = t \bmod 2$$

| $t$ | 0 | 0.5 | 1 | 1.5 | 2 |
|---|---|---|---|---|---|
| $\lvert x-1\rvert - 1$ | 0 | −0.5 | −1 | −0.5 | 0 |
| $2\lvert x-1\rvert - 1$ | 1 | 0 | −1 | 0 | 1 |

**Verallgemeinerung auf beliebige Periode $P$:** Die „1“ ist $P/2$ und der Faktor „2“ ist $4/P$:

$$a(t) = 4\left|\frac{t \bmod P}{P} - \frac12\right| - 1$$

Für $P = 2$ ist das genau die Formel oben. `np.mod` liefert auch für negative $t$ ein
Ergebnis in $[0, P)$, die Formel gilt also für alle $t$.

In [ ]:
def triangle_abs_mod(t, period=P):
    """Triangle signal from absolute value and modulo (idea: Emil)."""
    x = np.mod(t, period)
    return 4 * np.abs(x / period - 0.5) - 1

#### Variante 2: lineare Interpolation zwischen den Eckpunkten

**Idee:** Ein Dreieck ist eine stückweise lineare Funktion. Es ist durch seine Eckpunkte
vollständig bestimmt: $(0, 1)$ und $(P/2, -1)$, danach wiederholt es sich.
`np.interp` interpoliert linear zwischen gegebenen Punkten und kann mit `period=` periodisch
fortsetzen. Zwischen $P/2$ und $P$ interpoliert es dann automatisch zurück zum Eckpunkt
bei $P \equiv 0$.

Warum diese Variante:
- Sie beschreibt das Signal so, wie es in der Abbildung aussieht (Eckpunkte), ohne Formel herleiten zu müssen.
- Sie lässt sich direkt auf andere stückweise lineare Signale übertragen (Sägezahn, Trapez, …) — nur die Eckpunkte ändern sich. Das ist praktisch für Aufgabe 3.
- Sie rechnet völlig anders als Variante 1 und ist damit eine unabhängige Kontrolle.

Verworfene Alternative: $a(t) = \frac{2}{\pi}\arcsin\!\big(\cos(2\pi t/P)\big)$ ist ein eleganter
Einzeiler, verliert aber direkt an den Spitzen Genauigkeit: Dort ist $\cos \approx 1$, und
$\arcsin$ hat bei 1 eine unendliche Steigung. Nachgerechnet: Fehler bis ca. $7\cdot10^{-9}$ für
$t$ nahe 0, statt $10^{-16}$.

In [ ]:
def triangle_interp(t, period=P):
    """Triangle signal by periodic linear interpolation between its corners."""
    corners_t = [0.0, period / 2]
    corners_a = [1.0, -1.0]
    return np.interp(t, corners_t, corners_a, period=period)

#### Tests ausführen und Varianten vergleichen

In [ ]:
check_sample_times()
check_triangle(triangle_abs_mod)
check_triangle(triangle_interp)

# both variants on a fine grid over several periods
t_fine = np.linspace(-4, 12, 100_001)
difference = np.max(np.abs(triangle_abs_mod(t_fine) - triangle_interp(t_fine)))
print(f"all tests passed, max difference between the variants: {difference:.1e}")

# from here on the triangle is computed with variant 1
triangle = triangle_abs_mod

In [ ]:
# Plot des Signals (soll wie die Abbildung in der Angabe aussehen)
t_plot = np.linspace(0, T, 2001)
t_j = sample_times(N)
a_j = triangle(t_j)

fig, ax = plt.subplots()
ax.plot(t_plot, triangle_abs_mod(t_plot), label="Variante 1: Betrag + Modulo")
ax.plot(t_plot, triangle_interp(t_plot), "--", label="Variante 2: Interpolation")
ax.plot(t_j, a_j, ".", color="k", label=f"Abtastpunkte $a_j$, N = {N}")
ax.set_xlabel("t [ms]")
ax.set_ylabel("a(t)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3)
plt.show()

## 1 DFT

### 1a) Naive DFT und inverse DFT

$$
\hat a_k = \sum_{j=0}^{N-1} a_j\, e^{-2\pi i\, jk/N},
\qquad
a_j = \frac{1}{N}\sum_{k=0}^{N-1} \hat a_k\, e^{+2\pi i\, jk/N}
$$

Visualisieren: das Spektrum $\tilde a(\omega)$ und die Zerlegung in einzelne Fourier-Komponenten.

#### Tests zuerst

Die Tests vergleichen **nicht** mit `numpy.fft` (das ist Aufgabe 1b), sondern mit Ergebnissen,
die man von Hand ausrechnen kann. So prüfen sie die Definition selbst.

| Test | Signal $a_j$ | erwartet $\hat a_k$ |
|---|---|---|
| Handrechnung, $N = 4$ | $(1, 2, 3, 4)$ | $(10,\ -2+2i,\ -2,\ -2-2i)$ |
| Delta-Impuls | $(1, 0, \dots, 0)$ | alle $\hat a_k = 1$ |
| Konstante | alle $a_j = 1$ | $\hat a_0 = N$, sonst 0 |
| eine komplexe Schwingung | $e^{2\pi i\, m j/N}$ | $\hat a_m = N$, sonst 0 |
| Kosinus | $\cos(2\pi m j/N)$ | $\hat a_m = \hat a_{N-m} = N/2$, sonst 0 |
| Linearität | $\alpha a + \beta b$ | $\alpha \hat a + \beta \hat b$ |
| Rücktransformation | Zufallssignal (komplex) | `idft(dft(a)) == a` |

Handrechnung für $N = 4$: $e^{-2\pi i\, jk/4} = (-i)^{jk}$, also z. B.
$\hat a_1 = 1 + 2(-i) + 3(-1) + 4(i) = -2 + 2i$.

In [ ]:
def check_dft(dft, idft):
    """Tests for an implementation of the DFT and the inverse DFT."""
    # hand calculation, N = 4
    assert np.allclose(dft([1, 2, 3, 4]), [10, -2 + 2j, -2, -2 - 2j])
    assert np.allclose(idft([10, -2 + 2j, -2, -2 - 2j]), [1, 2, 3, 4])

    n = 16
    j = np.arange(n)

    # delta impulse -> all ones
    delta = np.zeros(n)
    delta[0] = 1
    assert np.allclose(dft(delta), np.ones(n))

    # constant -> only k = 0
    expected = np.zeros(n)
    expected[0] = n
    assert np.allclose(dft(np.ones(n)), expected)

    # one complex oscillation with frequency index m -> only k = m
    m = 3
    expected = np.zeros(n)
    expected[m] = n
    assert np.allclose(dft(np.exp(2j * np.pi * m * j / n)), expected)

    # cosine -> k = m and k = n - m, each n/2
    expected = np.zeros(n)
    expected[m] = expected[n - m] = n / 2
    assert np.allclose(dft(np.cos(2 * np.pi * m * j / n)), expected)

    # linearity
    rng = np.random.default_rng(0)
    a = rng.normal(size=n) + 1j * rng.normal(size=n)
    b = rng.normal(size=n)
    assert np.allclose(dft(2 * a - 3 * b), 2 * dft(a) - 3 * dft(b))

    # back transformation, also for odd n
    for n_test in (1, 7, 16, 64):
        a = rng.normal(size=n_test) + 1j * rng.normal(size=n_test)
        assert np.allclose(idft(dft(a)), a)

#### Implementierung

Direkt nach der Definition: Für jedes $k$ wird die Summe über $j$ ausgerechnet. Die Schleife
läuft über $k$, die Summe über $j$ macht numpy auf einmal. Aufwand: $N$ Summen mit je $N$ Termen,
also $\mathcal{O}(N^2)$.

DFT und inverse DFT unterscheiden sich nur im Vorzeichen des Exponenten und im Faktor $1/N$.
Deshalb rechnet eine gemeinsame Hilfsfunktion die Summe.

Detail: Im Exponenten steht $(jk \bmod N)$ statt $jk$. Mathematisch ist das dasselbe, weil
$e^{-2\pi i\, jk/N}$ in $jk$ periodisch mit Periode $N$ ist. Numerisch hält es das Argument der
Exponentialfunktion klein: Für $N = 4096$ wird $jk$ bis zu $1.7\cdot10^7$, und $\sin$/$\cos$
eines großen Arguments sind ungenauer (siehe `THEORIE.md`, Abschnitt 3).

In [ ]:
def _fourier_sum(values, sign):
    """sum_j values_j * exp(sign * 2 pi i j k / n) for every k = 0 ... n-1."""
    values = np.asarray(values, dtype=complex)
    n = len(values)
    j = np.arange(n)
    result = np.empty(n, dtype=complex)
    for k in range(n):
        phase = (j * k) % n  # same exponential, but a small argument
        result[k] = np.sum(values * np.exp(sign * 2j * np.pi * phase / n))
    return result


def dft(a):
    """Naive DFT: a_hat_k = sum_j a_j exp(-2 pi i j k / N)."""
    return _fourier_sum(a, sign=-1)


def idft(a_hat):
    """Naive inverse DFT: a_j = 1/N sum_k a_hat_k exp(+2 pi i j k / N)."""
    return _fourier_sum(a_hat, sign=+1) / len(a_hat)

In [ ]:
check_dft(dft, idft)
print("all DFT tests passed")

#### Spektrum $\tilde a(\omega)$

Mit $\omega_k = 2\pi k / T$ gilt $\tilde a(\omega_k) \approx \Delta t \cdot \hat a_k$
(Rechteckregel für das Fourier-Integral, siehe `THEORIE.md`, Abschnitt 2).

Die Indizes $k > N/2$ gehören zu negativen Frequenzen $(k - N)/T$. Die Hilfsfunktion
`angular_frequencies` ordnet jedem $k$ seine Kreisfrequenz zu; `np.fft.fftshift` sortiert
dann von negativ nach positiv. Zeit in ms, also $\omega$ in rad/ms und $\tilde a$ in ms.

In [ ]:
def angular_frequencies(n, window=T):
    """omega_k = 2 pi k / window, with k >= n/2 mapped to the negative frequency k - n.

    For even n the Nyquist index k = n/2 counts as negative, like in np.fft.fftfreq.
    """
    k = np.arange(n)
    k = np.where(k < (n + 1) // 2, k, k - n)
    return 2 * np.pi * k / window


# tests: n = 4 -> k = 0, 1, -2, -1 ; n = 5 -> k = 0, 1, 2, -2, -1 ; same as numpy
assert np.allclose(angular_frequencies(4, 2 * np.pi), [0, 1, -2, -1])
assert np.allclose(angular_frequencies(5, 2 * np.pi), [0, 1, 2, -2, -1])
for n_test in (4, 5, 64):
    assert np.allclose(angular_frequencies(n_test, 3.0), 2 * np.pi * np.fft.fftfreq(n_test, 3.0 / n_test))

dt = T / N
t_j = sample_times(N)
a_j = triangle(t_j)
a_hat = dft(a_j)
omega = angular_frequencies(N)
a_tilde = dt * a_hat

# the non-zero coefficients (positive frequencies)
print(f"{'k':>3} {'f [kHz]':>8} {'omega [rad/ms]':>15} {'ã(ω) [ms]':>12}")
for k in range(N // 2 + 1):
    if abs(a_hat[k]) > 1e-10 * np.max(np.abs(a_hat)):
        print(f"{k:>3} {k / T:>8.3f} {omega[k]:>15.3f} {a_tilde[k].real:>12.5f}")
print(f"largest imaginary part: {np.max(np.abs(a_hat.imag)):.1e}")

In [ ]:
order = np.fft.fftshift(np.arange(N))  # negative -> positive frequencies
omega_0 = 2 * np.pi / P                # fundamental angular frequency

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.stem(omega[order], a_tilde[order].real, basefmt=" ", label="Re ã(ω)")
ax1.plot(omega[order], a_tilde[order].imag, "x", color="C3", label="Im ã(ω)")
ax1.set_xlabel("ω [rad/ms]")
ax1.set_ylabel("ã(ω) [ms]")
ax1.set_title(f"Spektrum, N = {N}")
ax1.legend()

positive = np.arange(1, N // 2)  # k = N/2 (Nyquist) counts as negative frequency
ax2.semilogy(omega[positive], np.abs(a_tilde[positive]), "o")
for n_harmonic in (1, 3, 5, 7):
    ax2.axvline(n_harmonic * omega_0, color="gray", linestyle=":", zorder=0)
ax2.set_ylim(1e-18, 10)
ax2.set_xlim(0, 2 * np.pi * f_nyquist * 1.05)
ax2.set_xlabel("ω [rad/ms]")
ax2.set_ylabel("|ã(ω)| [ms]")
ax2.set_title("Betrag, logarithmisch (grau: n·ω₀, n ungerade)")

plt.tight_layout()
plt.show()

#### Zerlegung in Fourier-Komponenten

Für ein reelles Signal gehören $k$ und $N - k$ zusammen und ergeben eine reelle Kosinusschwingung
(siehe `THEORIE.md`, Abschnitt 2):

$$
\frac{1}{N}\left(\hat a_k e^{2\pi i\, k t/T} + \hat a_{N-k} e^{-2\pi i\, k t/T}\right)
= \frac{2}{N}\,\mathrm{Re}\!\left(\hat a_k\, e^{2\pi i\, k t/T}\right), \qquad 0 < k < N/2
$$

$k = 0$ (Mittelwert) und $k = N/2$ (Nyquist) haben keinen Partner und zählen einfach.
Die Komponenten werden auf einem feinen Zeitgitter ausgewertet, damit man die Schwingungen
zwischen den Abtastpunkten sieht.

Test: Die Summe **aller** Komponenten muss an den Abtastpunkten wieder genau $a_j$ ergeben.

In [ ]:
def fourier_component(a_hat, k, t, window=T):
    """Real oscillation belonging to frequency index k (0 <= k <= N/2) of a real signal."""
    n = len(a_hat)
    wave = a_hat[k] * np.exp(2j * np.pi * k * t / window) / n
    if k == 0 or 2 * k == n:  # mean value and Nyquist frequency have no partner
        return wave.real
    return 2 * wave.real


# test: all components together give back the samples (also for odd N and a random signal)
for a_test in (a_j, np.random.default_rng(1).normal(size=15)):
    n_test = len(a_test)
    t_test = sample_times(n_test)
    a_hat_test = dft(a_test)
    total = sum(fourier_component(a_hat_test, k, t_test) for k in range(n_test // 2 + 1))
    assert np.allclose(total, a_test)
print("decomposition test passed")

In [ ]:
t_plot = np.linspace(0, T, 2001)
harmonics = [k for k in range(1, N // 2 + 1) if abs(a_hat[k]) > 1e-10 * np.max(np.abs(a_hat))]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# individual components
for k in harmonics:
    ax1.plot(t_plot, fourier_component(a_hat, k, t_plot),
             label=f"k = {k} ({k / T:.1f} kHz), Amplitude {2 * abs(a_hat[k]) / N:.4f}")
ax1.set_ylabel("Komponente")
ax1.set_title("Einzelne Fourier-Komponenten")
ax1.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8)

# partial sums
partial = np.zeros_like(t_plot)
for count, k in enumerate(harmonics, start=1):
    partial = partial + fourier_component(a_hat, k, t_plot)
    ax2.plot(t_plot, partial, label=f"Summe bis k = {k} ({count} Komponente{'n' if count > 1 else ''})")
ax2.plot(t_j, a_j, ".", color="k", label="Abtastpunkte $a_j$")
ax2.set_xlabel("t [ms]")
ax2.set_ylabel("a(t)")
ax2.set_title("Teilsummen nähern sich dem Dreieck an")
ax2.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8)

plt.tight_layout()
plt.show()

#### Beobachtungen

- **Nur vier Linien** (und ihre Spiegelbilder bei negativem $\omega$): $k = 4, 12, 20, 28$, also
  0.5, 1.5, 2.5 und 3.5 kHz. Das sind die ungeraden Vielfachen der Grundfrequenz 500 Hz, wie von
  der Fourier-Reihe erwartet. Alle anderen $|\hat a_k|$ liegen bei $10^{-16}$, das ist Rundung.
- **Das Spektrum ist reell** (größter Imaginärteil $3.6\cdot10^{-15}$) und symmetrisch,
  $\tilde a(-\omega) = \tilde a(\omega)$. Grund: Das Dreieck ist reell und gerade (Aufgabe 2b).
- **Amplituden** der Kosinusschwingungen $2|\hat a_k|/N$ im Vergleich zur Fourier-Reihe $8/(\pi^2 n^2)$:

  | $k$ | $n$ | DFT | Fourier-Reihe | Abweichung |
  |---:|---:|---:|---:|---:|
  | 4 | 1 | 0.8211 | 0.8106 | 1.3 % |
  | 12 | 3 | 0.1012 | 0.0901 | 12 % |
  | 20 | 5 | 0.0452 | 0.0324 | 39 % |
  | 28 | 7 | 0.0325 | 0.0165 | 96 % |

  Je näher an der Nyquist-Frequenz ($k = 32$), desto größer die Abweichung. Das ist das
  Aliasing aus `THEORIE.md`, Abschnitt 4, und wird in 1d genauer untersucht.
- **Zerlegung:** Schon die Grundschwingung allein sieht dem Dreieck ähnlich, nur die Spitzen
  fehlen (Amplitude 0.82 statt 1). Jede weitere Komponente schärft die Spitzen nach.
  Mit allen vier Komponenten geht die Summe exakt durch alle 64 Abtastpunkte, weil die DFT an den
  Abtastpunkten exakt invertierbar ist. **Zwischen** den Abtastpunkten ist sie aber nicht das
  Dreieck: Mit 4 Kosinusschwingungen lässt sich kein scharfer Knick darstellen.

### 1b) Vergleich mit `numpy.fft`

Zuerst für das Dreieck, dann systematisch für verschiedene $N$.

**Fehlermaß.** Viele $\hat a_k$ sind exakt null und werden als $\sim 10^{-15}$ berechnet; ein relativer
Fehler pro Koeffizient wäre dort sinnlos. Deshalb der relative Fehler des ganzen Vektors:

$$
\text{Abweichung} = \frac{\lVert \texttt{dft}(a) - \texttt{np.fft.fft}(a) \rVert}{\lVert \texttt{np.fft.fft}(a) \rVert}
$$

Zur Einordnung: Eine Gleitkommazahl hat die relative Genauigkeit $\varepsilon \approx 2.2\cdot10^{-16}$
(`np.finfo(float).eps`).

**Test:** Beide Ergebnisse müssen bis auf Rundung übereinstimmen.

In [ ]:
eps = np.finfo(float).eps


def relative_difference(x, reference):
    """||x - reference|| / ||reference|| (2-norm over the whole vector)."""
    return np.linalg.norm(x - reference) / np.linalg.norm(reference)


a_hat_numpy = np.fft.fft(a_j)
print(f"triangle, N = {N}")
print(f"  max |dft - np.fft.fft|     = {np.max(np.abs(a_hat - a_hat_numpy)):.1e}"
      f"   (largest |â_k| = {np.max(np.abs(a_hat_numpy)):.1f})")
print(f"  relative difference         = {relative_difference(a_hat, a_hat_numpy):.1e}")
print(f"  max |idft - np.fft.ifft|    = {np.max(np.abs(idft(a_hat) - np.fft.ifft(a_hat_numpy))):.1e}")
print(f"  machine precision eps       = {eps:.1e}")

# test: equal up to rounding errors
assert relative_difference(a_hat, a_hat_numpy) < 100 * eps
assert np.allclose(idft(a_hat), np.fft.ifft(a_hat_numpy))

#### Abweichung in Abhängigkeit von $N$

Zufälliges komplexes Signal für $N = 2^2 \dots 2^{12}$. Gemessen werden:

1. **DFT gegen numpy:** relative Abweichung wie oben.
2. **Rundreise-Fehler:** `idft(dft(a))` gegen `a` bzw. `ifft(fft(a))` gegen `a`. Das braucht keine
   Referenz: Das exakte Ergebnis ist bekannt (das Ausgangssignal). Damit sieht man, welche Methode
   genauer ist, statt nur, dass sie sich unterscheiden.
3. **Dasselbe ohne den Modulo-Trick** im Exponenten, also mit $e^{-2\pi i\, jk/N}$ statt
   $e^{-2\pi i\, (jk \bmod N)/N}$. Das zeigt, woher ein Teil der Rundungsfehler kommt.

In [ ]:
def _fourier_sum_without_mod(values, sign):
    """Like _fourier_sum, but with the full argument j*k in the exponential."""
    values = np.asarray(values, dtype=complex)
    n = len(values)
    j = np.arange(n)
    result = np.empty(n, dtype=complex)
    for k in range(n):
        result[k] = np.sum(values * np.exp(sign * 2j * np.pi * j * k / n))
    return result


rng = np.random.default_rng(0)
n_values_1b = 2 ** np.arange(2, 13)
results_1b = {"dft vs numpy": [], "round trip dft": [], "round trip numpy": [],
              "dft without mod vs numpy": [], "round trip dft without mod": []}

for n in n_values_1b:
    x = rng.normal(size=n) + 1j * rng.normal(size=n)
    x_hat_numpy = np.fft.fft(x)
    x_hat = dft(x)
    x_hat_no_mod = _fourier_sum_without_mod(x, -1)

    results_1b["dft vs numpy"].append(relative_difference(x_hat, x_hat_numpy))
    results_1b["round trip dft"].append(relative_difference(idft(x_hat), x))
    results_1b["round trip numpy"].append(relative_difference(np.fft.ifft(x_hat_numpy), x))
    results_1b["dft without mod vs numpy"].append(relative_difference(x_hat_no_mod, x_hat_numpy))
    results_1b["round trip dft without mod"].append(
        relative_difference(_fourier_sum_without_mod(x_hat_no_mod, +1) / n, x))

print(f"{'N':>6}" + "".join(f"{name:>28}" for name in results_1b))
for i, n in enumerate(n_values_1b):
    print(f"{n:>6}" + "".join(f"{values[i]:>28.1e}" for values in results_1b.values()))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
styles = ["o-", "s-", "^-", "o--", "s--"]
for (name, values), style in zip(results_1b.items(), styles):
    ax.loglog(n_values_1b, values, style, label=name)
ax.loglog(n_values_1b, eps * n_values_1b / 10, ":", color="gray", label=r"$\propto N$")
ax.axhline(eps, color="k", linestyle=":", label=r"$\varepsilon$")
ax.set_xlabel("N")
ax.set_ylabel("relativer Fehler")
ax.set_title("DFT gegen numpy.fft: Rundungsfehler")
ax.legend(fontsize=8, loc="upper left")
plt.show()

#### Beobachtungen

- **Dreieck, $N = 64$:** größte Abweichung $5.7\cdot10^{-15}$ bei Werten bis 26.3, relativ
  $2.9\cdot10^{-16}$. Das ist etwa ein $\varepsilon$: beide Ergebnisse stimmen bis auf die letzte
  Stelle überein.
- **Mit Modulo-Trick** bleiben alle Fehler bis $N = 4096$ bei $(1 \dots 4)\cdot10^{-16}$, also nahe
  $\varepsilon$. Die naive DFT ist hier **genauso genau** wie numpys FFT (Rundreise $3.9\cdot10^{-16}$
  gegen $3.5\cdot10^{-16}$).
- **Ohne Modulo-Trick** wächst der Fehler proportional zu $N$, bei $N = 4096$ auf $8\cdot10^{-13}$,
  also 2000-mal schlechter. Der Unterschied steckt nur im Argument der Exponentialfunktion.

**Erwartet hatte ich** (und so steht es zunächst in `THEORIE.md`), dass sich bei der naiven DFT die
Rundungsfehler der $N$ Summanden ansammeln und sie deshalb ungenauer als die FFT ist. Das ist hier nicht
der Fall. Erklärung:

1. **Summation:** `np.sum` addiert laut numpy-Dokumentation paarweise (pairwise summation): erst
   Paare, dann Paare von Paaren usw. Der Rundungsfehler wächst dadurch nur mit $\log N$, nicht mit $N$ –
   wie bei der FFT. Mit einer einfachen Python-Schleife `s += ...` wäre das anders.
2. **Exponentialfunktion:** Das Argument $2\pi\, jk/N$ wird vor dem Aufruf von `np.exp` gerundet,
   mit einem absoluten Fehler von etwa $\varepsilon \cdot |\text{Argument}|$. Ohne Modulo ist das
   Argument bis zu $2\pi N$ groß, der Phasenfehler also $\propto N$. Mit Modulo bleibt es unter $2\pi$.

`THEORIE.md` wurde entsprechend korrigiert.

> **Frage:** Warum ist die Differenz nicht exakt null?
>
> **Antwort:** Weil Gleitkommazahlen nur etwa 16 Stellen haben und jede Rechenoperation das Ergebnis
> rundet (relativer Fehler bis $\varepsilon/2 \approx 1.1\cdot10^{-16}$). Schon die Faktoren
> $e^{-2\pi i\, jk/N}$ sind nicht exakt darstellbar, und $\pi$ selbst ist gerundet.
>
> DFT und FFT berechnen mathematisch dasselbe, aber auf verschiedenen Wegen: die DFT als $N$ Summen
> über $N$ Produkte, die FFT in $\log_2 N$ Stufen von Butterflies mit anderen Zwischenergebnissen.
> Gleitkomma-Addition ist nicht assoziativ, $(a + b) + c \ne a + (b + c)$ im Allgemeinen.
> Verschiedene Rechenwege runden deshalb verschieden, und die Ergebnisse unterscheiden sich in den
> letzten Stellen.
>
> Die gemessene Abweichung ist relativ $\sim 3\cdot10^{-16}$, also ein bis zwei $\varepsilon$. Das ist
> die bestmögliche Übereinstimmung, kein Fehler einer der beiden Implementierungen. Ein Fehler in der
> Implementierung würde sich als Abweichung von $10^{-3}$ oder mehr zeigen.
>
> Wie groß der Rundungsfehler ist, hängt von der Implementierung ab: Ohne den Modulo-Trick im Exponenten
> wächst er $\propto N$.

### 1c) Laufzeit DFT gegen FFT für verschiedene $N$

Erwartung (`THEORIE.md`, Abschnitt 3): DFT $\mathcal{O}(N^2)$, FFT $\mathcal{O}(N \log N)$.

**Zeitmessung.** `time.perf_counter()` vor und nach dem Aufruf. Zwei Probleme und ihre Lösung:
- Ein FFT-Aufruf dauert nur Mikrosekunden, zu kurz für eine einzelne Messung. Deshalb wird der Aufruf so
  oft hintereinander wiederholt (`number`, jeweils verdoppelt), bis eine Messung lange genug dauert,
  und die Zeit durch `number` geteilt.
- Andere Programme auf dem Rechner können eine Messung nur verlangsamen, nie beschleunigen. Deshalb
  mehrere Messungen (`repeats`) und davon das **Minimum**.

**Test der Zeitmessung:** an etwas mit bekannter Dauer, `time.sleep(0.01)`.

In [ ]:
def time_call(func, *args, min_time=0.1, repeats=5):
    """Runtime of one call func(*args) in seconds (best of `repeats` measurements)."""
    # number of calls per measurement, so that one measurement takes at least min_time / repeats
    number = 1
    while True:
        start = time.perf_counter()
        for _ in range(number):
            func(*args)
        elapsed = time.perf_counter() - start
        if elapsed >= min_time / repeats:
            break
        number *= 2
    best = elapsed / number
    for _ in range(repeats - 1):
        start = time.perf_counter()
        for _ in range(number):
            func(*args)
        best = min(best, (time.perf_counter() - start) / number)
    return best


# test: sleeping 10 ms must be measured as 10 ms per call (not the sum over all repetitions)
measured = time_call(time.sleep, 0.01)
assert 0.01 <= measured < 0.02, measured
print(f"time_call test passed: sleep(0.01) measured as {measured * 1e3:.2f} ms")

In [ ]:
rng = np.random.default_rng(0)
n_dft = 2 ** np.arange(2, 13)    # naive DFT: 4 ... 4096 (4096 takes about 0.5 s per call)
n_fft = 2 ** np.arange(2, 23)    # FFT: 4 ... 4 194 304

times_dft = np.array([time_call(dft, rng.normal(size=n)) for n in n_dft])
times_fft = np.array([time_call(np.fft.fft, rng.normal(size=n)) for n in n_fft])

print(f"{'N':>9} {'DFT':>11} {'FFT':>11} {'DFT / FFT':>10}")
for i, n in enumerate(n_fft):
    if i < len(n_dft):
        print(f"{n:>9} {times_dft[i]:>10.2e}s {times_fft[i]:>10.2e}s {times_dft[i] / times_fft[i]:>10.0f}")
    else:
        print(f"{n:>9} {'':>11} {times_fft[i]:>10.2e}s")

**Steigungen.** Im log-log-Plot ist eine Laufzeit $\propto N^p$ eine Gerade mit Steigung $p$. Die
Steigung wird nur für große $N$ gefittet, weil bei kleinen $N$ ein fester Aufwand pro Aufruf dominiert.

In [ ]:
def loglog_slope(x, y):
    """Slope of the straight line through log(y) versus log(x)."""
    return np.polyfit(np.log(x), np.log(y), 1)[0]


large_dft = n_dft >= 1024
large_fft = n_fft >= 2**14
slope_dft = loglog_slope(n_dft[large_dft], times_dft[large_dft])
slope_fft = loglog_slope(n_fft[large_fft], times_fft[large_fft])
print(f"DFT: time ~ N^{slope_dft:.2f}  (N >= 1024)")
print(f"FFT: time ~ N^{slope_fft:.2f}  (N >= 2^14), N log N would give slightly more than 1")

# runtime per N log2 N for the FFT: constant if the FFT is O(N log N)
per_op = times_fft[large_fft] / (n_fft[large_fft] * np.log2(n_fft[large_fft]))
print(f"FFT time / (N log2 N): {per_op.min() * 1e9:.2f} ... {per_op.max() * 1e9:.2f} ns")

# extrapolation of the DFT to the largest FFT size
n_big = n_fft[-1]
dft_big = times_dft[-1] * (n_big / n_dft[-1]) ** 2
print(f"\nDFT for N = {n_big}: extrapolated {dft_big:.1e} s = {dft_big / 86400:.1f} days, "
      f"FFT measured {times_fft[-1]:.2f} s")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.loglog(n_dft, times_dft, "o-", label=f"naive DFT, Steigung {slope_dft:.2f}")
ax1.loglog(n_fft, times_fft, "s-", label=f"np.fft.fft, Steigung {slope_fft:.2f}")
# reference lines through the last measured point
ax1.loglog(n_dft, times_dft[-1] * (n_dft / n_dft[-1]) ** 2, ":", color="C0", label=r"$\propto N^2$")
ref = n_fft * np.log2(n_fft)
ax1.loglog(n_fft, times_fft[-1] * ref / ref[-1], ":", color="C1", label=r"$\propto N \log N$")
ax1.set_xlabel("N")
ax1.set_ylabel("Laufzeit pro Aufruf [s]")
ax1.set_title("Laufzeit")
ax1.legend()

ax2.loglog(n_dft, times_dft / times_fft[: len(n_dft)], "o-", color="k")
ax2.set_xlabel("N")
ax2.set_ylabel("Laufzeit DFT / Laufzeit FFT")
ax2.set_title("Wie viel schneller ist die FFT?")

plt.tight_layout()
plt.show()

#### Beobachtungen

Die Zahlen stammen aus einem Lauf auf diesem Rechner; die Laufzeiten schwanken von Lauf zu Lauf um
einige Prozent, die Steigungen kaum.

| $N$ | DFT | FFT | DFT / FFT |
|---:|---:|---:|---:|
| 4 | 28 µs | 4.0 µs | 7 |
| 64 | 0.54 ms | 4.7 µs | 115 |
| 1024 | 35 ms | 13 µs | 2 600 |
| 4096 | 0.48 s | 47 µs | 10 000 |
| 4 194 304 | ≈ 6 Tage (hochgerechnet) | 0.11 s | ≈ 5 000 000 |

- **DFT, Steigung 1.89 für $N \ge 1024$:** nahe an 2. Für kleine $N$ ist die Steigung nur etwa 1: Dort
  dominiert der feste Aufwand pro Schleifendurchlauf über $k$ (ca. 7 µs: Array anlegen, `np.exp`
  aufrufen), und davon gibt es $N$ Stück. Erst ab etwa $N = 1000$ überwiegt die eigentliche Arbeit
  $\propto N^2$.
- **FFT, Steigung 1.12 für $N \ge 2^{14}$:** $N \log N$ hat im log-log-Plot keine konstante Steigung,
  sondern $1 + 1/\ln N$, im Bereich $2^{14} \dots 2^{22}$ also etwa 1.07–1.10. Besser sieht man es an
  Laufzeit$/(N \log_2 N)$: konstant etwa 1 ns.
- **FFT bei kleinem $N$:** bis $N \approx 256$ fast konstant 4–6 µs. Das ist der feste Aufwand eines
  numpy-Aufrufs, nicht die Rechnung.
- Ein Messwert (FFT bei $N = 512$) war in einem früheren Lauf 4-mal zu langsam und in Wiederholungen
  nicht reproduzierbar: eine durch andere Prozesse gestörte Messung. Deshalb jetzt das Minimum aus
  5 statt 3 Messungen.

> **Frage:** Wie skalieren die Laufzeiten, und um welchen Faktor ist die FFT schneller?
>
> **Antwort:** Die naive DFT skaliert wie $N^2$ (gemessene Steigung 1.89 für große $N$), die FFT wie
> $N \log N$ (Laufzeit$/(N\log_2 N) \approx$ 1 ns konstant). Der Faktor ist deshalb nicht fest, sondern
> wächst etwa wie $N/\log N$: 7 bei $N = 4$, 115 bei $N = 64$, 10 000 bei $N = 4096$.
> Für $N = 4\,194\,304$ bräuchte die naive DFT hochgerechnet etwa 6 Tage, die FFT 0.11 s.
>
> Der Vergleich ist nicht ganz fair: Die DFT ist eine Python-Schleife (mit numpy innen), die FFT
> kompilierter C-Code. Ein Teil des Faktors ist Python gegen C. Bei großem $N$ dominiert aber der
> Unterschied $N^2$ gegen $N \log N$, und der lässt sich durch keine Implementierung wettmachen.
> Den fairen Vergleich (beide in Python) liefert Aufgabe 2e mit der eigenen rekursiven FFT.

### 1d) Vergleich mit dem analytischen Ergebnis

Fourier-Reihe des Dreiecks (Grundfrequenz $f_0 = 1/P$):

$$
a(t) = \frac{8}{\pi^2} \sum_{n = 1, 3, 5, \dots} \frac{1}{n^2} \cos(2\pi n f_0 t)
= \sum_{m=-\infty}^{\infty} c_m\, e^{2\pi i\, m t/T}
$$

In der komplexen Form über das Fenster $T$ gehört $m$ zur Frequenz $m/T$. Die $n$-te Oberschwingung liegt
bei $m = \pm n\,T/P$, und jeder Kosinus teilt sich auf $+m$ und $-m$ auf:

$$
c_m = \frac{4}{\pi^2 n^2} \quad \text{für } m = \pm n\,T/P,\ n \text{ ungerade}, \qquad c_m = 0 \text{ sonst}
$$

Mit $T = 4P$: $c_{\pm 4} = 4/\pi^2$, $c_{\pm 12} = 4/(9\pi^2)$, …
Voraussetzung: $T$ ist ein ganzzahliges Vielfaches von $P$ (sonst Leakage, Aufgabe 3c).

**Erwartung:** $\hat a_k / N \approx c_k$. Exakt gilt wegen Aliasing (`THEORIE.md`, Abschnitt 4)

$$
\frac{\hat a_k}{N} = \sum_{l=-\infty}^{\infty} c_{k + lN}
$$

Ab hier wird `np.fft.fft` verwendet: Es liefert dasselbe wie `dft` (1b), ist aber schnell genug für große $N$.

#### Tests zuerst

Die analytischen Koeffizienten werden unabhängig von der DFT geprüft:
- bekannte Werte $c_4 = 4/\pi^2$, $c_{12} = 4/(9\pi^2)$, $c_{-4} = c_4$, $c_5 = c_8 = 0$
- negative Frequenzen: DFT-Index $k = N - 4$ gehört zu $m = -4$
- Summe aller $c_m$ = $a(0) = 1$ (Fourier-Reihe bei $t = 0$)
- Parseval für die Fourier-Reihe: $\sum_m |c_m|^2 = \frac{1}{T}\int_0^T a(t)^2\,dt = \frac13$ (Mittelwert von $a^2$ für ein Dreieck)
- andere Fensterlänge: $T = 2P$ → Grundschwingung bei $m = 2$

In [ ]:
def frequency_index(k, n_points):
    """Signed frequency index m for DFT index k: k >= n/2 belongs to the negative frequency k - n."""
    k = np.asarray(k)
    return np.where(k < (n_points + 1) // 2, k, k - n_points)


def fourier_coefficient(m, window=T, period=P):
    """Fourier series coefficient c_m of the triangle for the frequency m / window."""
    harmonic = np.abs(np.asarray(m, dtype=float)) * period / window  # n = |m| P / T
    n = np.round(harmonic)
    # rtol=0: the default relative tolerance (1e-5) would accept e.g. 1000000.3 as an integer
    is_odd_harmonic = np.isclose(harmonic, n, rtol=0, atol=1e-9) & (n % 2 == 1)
    return np.where(is_odd_harmonic, 4 / (np.pi**2 * np.maximum(n, 1) ** 2), 0.0)


def analytic_coefficients(k, n_points, window=T, period=P):
    """c_m for the DFT indices k (without aliasing)."""
    return fourier_coefficient(frequency_index(k, n_points), window, period)


def aliased_coefficients(k, n_points, window=T, period=P, n_alias=20000):
    """sum_l c_(m + l N): what the DFT should give exactly (a_hat_k / N), truncated at |l| <= n_alias."""
    m = frequency_index(k, n_points)
    l = np.arange(-n_alias, n_alias + 1)
    return fourier_coefficient(m[:, None] + l[None, :] * n_points, window, period).sum(axis=1)

In [ ]:
# Tests: analytische Koeffizienten
assert np.isclose(fourier_coefficient(4), 4 / np.pi**2)
assert np.isclose(fourier_coefficient(12), 4 / (9 * np.pi**2))
assert np.isclose(fourier_coefficient(-4), fourier_coefficient(4))
assert np.allclose(fourier_coefficient([0, 1, 5, 8, 16]), 0)

# negative frequencies: DFT index N - 4 is m = -4
assert np.isclose(analytic_coefficients(np.array([60]), 64), 4 / np.pi**2)
assert np.allclose(frequency_index(np.arange(4), 4), [0, 1, -2, -1])

# sum of all c_m = a(0) = 1 ; sum of |c_m|^2 = mean of a^2 = 1/3   (series truncated at |m| <= 4 * 10^6)
m_all = np.arange(-4_000_000, 4_000_001)
c_all = fourier_coefficient(m_all)
assert np.isclose(c_all.sum(), 1, atol=1e-6)
assert np.isclose(np.sum(c_all**2), 1 / 3, atol=1e-12)

# other window: T = 2P -> fundamental at m = 2
assert np.isclose(fourier_coefficient(2, window=2 * P), 4 / np.pi**2)
assert np.isclose(fourier_coefficient(4, window=2 * P), 0)
print("analytic coefficient tests passed")

#### Vergleich für $N = 64$

In [ ]:
k_all = np.arange(N)
dft_coeff = np.fft.fft(a_j).real / N          # â_k / N (the spectrum is real, see 1a)
analytic = analytic_coefficients(k_all, N)
aliased = aliased_coefficients(k_all, N)

# test: with aliasing the DFT is reproduced (up to the truncation of the alias sum)
assert np.allclose(dft_coeff, aliased, atol=1e-8)

lines = [k for k in range(1, N // 2) if analytic[k] > 0]
print(f"{'k':>3} {'f [kHz]':>8} {'â_k/N':>10} {'c_k':>10} {'Σ c_(k+lN)':>11} {'rel. error':>11}")
for k in lines:
    print(f"{k:>3} {k / T:>8.2f} {dft_coeff[k]:>10.5f} {analytic[k]:>10.5f} {aliased[k]:>11.5f} "
          f"{(dft_coeff[k] - analytic[k]) / analytic[k]:>10.1%}")
print(f"max |â_k/N - Σ c_(k+lN)| = {np.max(np.abs(dft_coeff - aliased)):.1e}")

In [ ]:
f_k = np.arange(1, N // 2) / T   # positive frequencies in kHz
idx = np.arange(1, N // 2)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.semilogy(f_k[analytic[idx] > 0], dft_coeff[idx][analytic[idx] > 0], "o", ms=9, label=r"DFT $\hat a_k / N$")
ax.semilogy(f_k[analytic[idx] > 0], analytic[idx][analytic[idx] > 0], "s", label=r"Fourier-Reihe $c_k$")
ax.semilogy(f_k[analytic[idx] > 0], aliased[idx][analytic[idx] > 0], "x", ms=10, color="k",
            label=r"mit Aliasing $\sum_l c_{k+lN}$")
f_cont = np.linspace(0.3, 4.2, 200)
ax.semilogy(f_cont, 4 / (np.pi**2 * (f_cont * P) ** 2), ":", color="C1", label=r"$4/(\pi^2 n^2)$, $n = f P$")
ax.axvline(f_nyquist, color="gray", linestyle="--", label="Nyquist-Frequenz")
ax.set_xlabel("f [kHz]")
ax.set_ylabel("Koeffizient")
ax.set_title(f"DFT gegen analytisches Ergebnis, N = {N}")
ax.legend()
plt.show()

#### Fehler in Abhängigkeit von der Frequenz, für verschiedene $N$

Der Fehler wird gegen $f / f_\text{Nyquist}$ aufgetragen. Wenn er nur vom Abstand zur Nyquist-Frequenz
abhängt, liegen die Kurven für alle $N$ übereinander.

**Vorhersage:** Für ungerade Oberschwingungen ist $c \propto 1/m^2$. Mit $x = f/f_\text{Nyquist} = 2k/N$
wird der relative Fehler

$$
\frac{\hat a_k/N - c_k}{c_k} = \sum_{l \ne 0} \frac{c_{k+lN}}{c_k} = \sum_{l \ne 0} \left(\frac{k}{k + lN}\right)^2
= \sum_{l \ne 0} \left(\frac{x}{x + 2l}\right)^2
$$

Er hängt also nur von $x$ ab. Bei $x \to 1$ ist der Term $l = -1$ gleich 1: Fehler über 100 %.
(Gilt, solange $N/4$ gerade ist, sodass $k + lN$ wieder eine ungerade Oberschwingung ist; hier $N \ge 8$.)

In [ ]:
def predicted_relative_error(x, n_alias=20000):
    """sum_(l != 0) (x / (x + 2 l))^2 for x = f / f_Nyquist."""
    l = np.concatenate([np.arange(-n_alias, 0), np.arange(1, n_alias + 1)])
    x = np.asarray(x, dtype=float)
    return np.sum((x[:, None] / (x[:, None] + 2 * l[None, :])) ** 2, axis=1)


fig, ax = plt.subplots(figsize=(9, 4.5))
for n, size in zip((16, 64, 256, 1024), (11, 8, 6, 4)):
    k = np.arange(n)
    a_hat_n = np.fft.fft(triangle(sample_times(n))).real / n
    c_n = analytic_coefficients(k, n)
    line_k = np.array([kk for kk in range(1, n // 2) if c_n[kk] > 0])
    x = 2 * line_k / n
    ax.semilogy(x, (a_hat_n[line_k] - c_n[line_k]) / c_n[line_k], "o", ms=size, label=f"N = {n}")
x_cont = np.linspace(0.01, 0.99, 200)
ax.semilogy(x_cont, predicted_relative_error(x_cont), "k-", lw=1, label="Vorhersage Aliasing")
ax.set_xlabel(r"$f / f_\mathrm{Nyquist}$")
ax.set_ylabel("relativer Fehler")
ax.set_title("Fehler der DFT gegen die Fourier-Reihe")
ax.legend()
plt.show()

#### Einfluss von $\Delta t$ und $T$

1. **$T = 8$ ms fest, $\Delta t$ variiert** (über $N$): Fehler der Grundschwingung (500 Hz) gegen $\Delta t$.
   Vorhersage: Für kleine $x = 2k/N$ wird $\sum_{l \ne 0} (x/(x+2l))^2 \approx \frac{x^2}{4}\sum_{l\ne0}\frac{1}{l^2} = \frac{x^2}{4}\cdot\frac{\pi^2}{3}$,
   also Fehler $\approx \frac{\pi^2}{3}\,(k\,\Delta t/T)^2 \propto \Delta t^2$.
2. **$\Delta t = 0.125$ ms fest, $T$ variiert** ($T = 2, 4, 8, 16$ ms, immer Vielfache von $P$):
   Spektrum $\Delta t\,|\hat a_k|$ über der Frequenz.

In [ ]:
# 1. fixed T, varying dt
n_values_dt = 2 ** np.arange(3, 13)
dt_values = T / n_values_dt
k_fund = int(T / P)  # index of the fundamental (500 Hz): k = T / P = 4
error_fund = np.array([np.fft.fft(triangle(sample_times(n))).real[k_fund] / n / fourier_coefficient(k_fund) - 1
                       for n in n_values_dt])
slope_dt = loglog_slope(dt_values, error_fund)

# 2. fixed dt, varying T
dt_fixed = 0.125
windows = [2.0, 4.0, 8.0, 16.0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.loglog(dt_values, error_fund, "o-", label=f"gemessen, Steigung {slope_dt:.2f}")
ax1.loglog(dt_values, np.pi**2 / 3 * (k_fund * dt_values / T) ** 2, "k:",
           label=r"Vorhersage $\frac{\pi^2}{3}\,(k\,\Delta t/T)^2$")
ax1.set_xlabel(r"$\Delta t$ [ms]")
ax1.set_ylabel("relativer Fehler bei 500 Hz")
ax1.set_title(f"T = {T:g} ms fest")
ax1.legend()

print(f"{'T [ms]':>7} {'N':>5} {'Δf [kHz]':>9} {'â/N at 500 Hz':>14} {'Δt·â at 500 Hz [ms]':>20}")
for i, window in enumerate(windows):
    n = int(round(window / dt_fixed))
    a_hat_w = np.fft.fft(triangle(sample_times(n, window))).real
    f_w = np.arange(n // 2) / window
    ax2.plot(f_w, dt_fixed * np.abs(a_hat_w[: n // 2]), "o", ms=4 + 2 * (len(windows) - i),
             label=f"T = {window:g} ms, N = {n}", zorder=10 - i)
    k_500 = int(round(0.5 * window))
    print(f"{window:>7g} {n:>5} {1 / window:>9.4f} {a_hat_w[k_500] / n:>14.5f} {dt_fixed * a_hat_w[k_500]:>20.4f}")
ax2.set_xlim(0, 3)
ax2.set_xlabel("f [kHz]")
ax2.set_ylabel(r"$\Delta t\,|\hat a_k|$ [ms]")
ax2.set_title(rf"$\Delta t$ = {dt_fixed} ms fest")
ax2.legend()

plt.tight_layout()
plt.show()
print(f"error at 500 Hz ~ dt^{slope_dt:.2f}")

#### Beobachtungen

- **Lage der Linien stimmt exakt:** DFT und Fourier-Reihe haben Linien nur bei 0.5, 1.5, 2.5, 3.5 kHz.
- **Höhe stimmt nur bei tiefen Frequenzen:** Bei $N = 64$ wächst der relative Fehler von 1.3 % (500 Hz)
  auf 96 % (3.5 kHz).
- **Die Aliasing-Formel erklärt den Fehler vollständig:** $\hat a_k/N$ und $\sum_l c_{k+lN}$ stimmen auf
  $1.6\cdot10^{-7}$ überein. Dieser Rest ist der Abbruch der Summe bei $|l| \le 20\,000$, kein Fehler
  der DFT.
- **Der Fehler hängt nur von $f/f_\text{Nyquist}$ ab:** Für $N = 16$ bis $1024$ liegen alle Punkte auf
  derselben Kurve, und diese Kurve ist die Vorhersage $\sum_{l\ne0}(x/(x+2l))^2$.
- **$\Delta t$ variiert ($T$ fest):** Der Fehler bei 500 Hz fällt wie $\Delta t^{2.05}$, passend zur
  Vorhersage $\frac{\pi^2}{3}(k\,\Delta t/T)^2$. Halber Abtastschritt → viermal kleinerer Fehler.
  Nur beim gröbsten $\Delta t = 1$ ms (N = 8) weicht es ab: dort liegt 500 Hz genau auf der
  Nyquist-Frequenz, und die Näherung für kleines $x$ gilt nicht mehr.
- **$T$ variiert ($\Delta t$ fest):** Die Linien bleiben bei denselben Frequenzen. $\hat a_k/N$ ist für
  alle $T$ gleich (0.41053), $\Delta t\,\hat a_k$ wächst proportional zu $T$ (0.82, 1.64, 3.28, 6.57 ms).
  Dazwischen kommen nur weitere Nullen hinzu, weil das Raster $\Delta f = 1/T$ feiner wird.
- **Gefundener Fehler beim Schreiben der Tests:** `np.isclose` hat standardmäßig eine relative Toleranz
  von $10^{-5}$. Bei Oberschwingungsnummern um $10^6$ galt damit fast jede Zahl als ganzzahlig, und die
  Summe aller $c_m$ kam auf $1.00003$ statt 1. Für die kleinen $m$, die die DFT braucht, wäre das nie
  aufgefallen. Korrigiert mit `rtol=0`.
- **Meine erste Vorhersage war falsch:** Ich hatte nur die Alias-Terme $l = \pm1$ berücksichtigt
  ($2\,(k\Delta t/T)^2$), das war um den Faktor 1.6 zu klein. Alle $l$ zusammen ergeben
  $\sum_{l\ne0} 1/l^2 = \pi^2/3$ statt 2.

> **Frage:** Wie beeinflussen $\Delta t$ und $T$ das FFT-Ergebnis?
>
> **Antwort:**
> - **$T$ (Fensterlänge) bestimmt die Frequenzauflösung** $\Delta f = 1/T$: Die DFT liefert nur die
>   Frequenzen $f_k = k/T$. Längeres Fenster → feineres Raster. Solange $T$ ein Vielfaches der Periode
>   ist, bleiben die Werte an den Linien gleich ($\hat a_k/N$ unverändert), dazwischen kommen Nullen.
>   Das kontinuierliche Spektrum $\Delta t\,\hat a_k \approx \tilde a(\omega_k)$ wächst dagegen $\propto T$:
>   Für ein unendlich langes periodisches Signal würden die Linien zu Delta-Peaks.
>   Ist $T$ kein Vielfaches der Periode, liegen die Linien zwischen den Rasterpunkten (Leakage, 3c).
> - **$\Delta t$ (Abtastschritt) bestimmt die höchste Frequenz** $f_\text{Nyquist} = 1/(2\Delta t)$ und
>   den Aliasing-Fehler. Bei festem $T$ ändert $\Delta t$ nichts an der Lage der Frequenzen, aber der
>   Fehler bei einer festen Frequenz fällt $\propto \Delta t^2$ (gemessen: Steigung 2.05).
>   Der Exponent 2 kommt vom Dreieck ($c \propto 1/m^2$); für ein Signal mit Sprung wäre er kleiner.

> **Frage:** Warum wächst der Fehler zur Nyquist-Frequenz hin?
>
> **Antwort:** Wegen Aliasing. An den Abtastpunkten sind die Frequenzen $m$ und $m + lN$ nicht zu
> unterscheiden, $e^{2\pi i (k+lN) j/N} = e^{2\pi i kj/N}$. Das Dreieck hat unendlich viele
> Oberschwingungen, und alle oberhalb der Nyquist-Frequenz werden auf tiefere Frequenzen zurückgefaltet:
> $\hat a_k / N = \sum_l c_{k+lN}$.
>
> Der größte Fremdbeitrag ist $c_{k-N}$, die Komponente bei der negativen Frequenz $k - N$.
> Bei kleinem $k$ ist $|k - N| \approx N \gg k$, also $c_{k-N} \ll c_k$. Nahe Nyquist ist
> $|k - N| \approx N/2 \approx k$: Der Fremdbeitrag ist **genauso groß** wie der gesuchte Wert.
> Relativ ergibt das $\sum_{l\ne0}(x/(x+2l))^2$ mit $x = f/f_\text{Nyquist}$, also etwa
> $\frac{\pi^2}{12}x^2$ für kleines $x$ und über 100 % bei $x \to 1$. Gemessen für $N = 16 \dots 1024$
> liegen alle Werte auf dieser Kurve.
>
> Kein Rundungsproblem: Der Fehler ist eine Eigenschaft der Abtastung, nicht der Rechnung. Die DFT
> berechnet die abgetasteten Daten exakt (bis auf $10^{-16}$), aber die Abtastung hat schon Information
> über die hohen Frequenzen verloren.

## 2 Eigenschaften der DFT (numerisch prüfen)

### 2a) Symmetrie für reelle Signale

$$\hat a_{N-k} = \hat a_k^{\,*}$$

In [ ]:
# TODO: Symmetrie für das Dreieck und für ein zufälliges reelles Signal prüfen

### 2b) Verschiebungssatz

Verschiebung um $m$ Punkte, $b_j = a_{(j-m) \bmod N}$ (`np.roll(a, m)`):

$$\hat b_k = \hat a_k\, e^{-2\pi i\, km/N}$$

In [ ]:
# TODO: Signal verschieben, mit dem Verschiebungssatz vergleichen

> **Frage:** Warum ist $\hat a_k$ für das Dreieck sogar rein reell?
>
> **Antwort:** *TODO*

### 2c) Rücktransformation und Parseval

$\mathcal{F} \circ \mathcal{F}^{-1}(a) = a$ und Parseval.

**Achtung:** Mit der verwendeten Konvention gilt $\sum_j |a_j|^2 = \frac{1}{N} \sum_k |\hat a_k|^2$.
In der Angabe fehlt der Faktor $1/N$ (siehe `THEORIE.md`, Abschnitt 5).

In [ ]:
# TODO: Rücktransformation und Parseval prüfen

### 2d) Cooley-Tukey-Rekursion

$$
\hat a_k = E_k + w_N^{-k} O_k, \qquad
\hat a_{k+N/2} = E_k - w_N^{-k} O_k
$$

$E_k = \text{DFT}(a_0, a_2, \dots)$, $O_k = \text{DFT}(a_1, a_3, \dots)$, $w_N = e^{2\pi i/N}$

In [ ]:
# TODO: E und O mit dft berechnen, zusammensetzen, mit dft(a) vergleichen

### 2e) Rekursive FFT (radix-2)

In [ ]:
def fft_recursive(a):
    # TODO: rekursive Radix-2-FFT, len(a) muss eine Zweierpotenz sein
    raise NotImplementedError

In [ ]:
# Tests: rekursive FFT
# TODO: Vergleich mit np.fft.fft für mehrere N, Fehler bei N keine Zweierpotenz

In [ ]:
# Laufzeit: dft, fft_recursive, np.fft.fft
# TODO

## 3 Andere Signale

Rechteck, Sägezahn, Sinus, Gauß-Puls, Rauschen, …

In [ ]:
# TODO: Signale definieren (gleiche Zeitpunkte wie das Dreieck)

In [ ]:
# Plot: Signale und ihre Spektren
# TODO

### 3a) Welche Spektren sind reell, imaginär, komplex?

In [ ]:
# TODO: Real- und Imaginärteil der Spektren auswerten

> **Frage:** Welche Spektren sind reell, imaginär, komplex, und warum?
>
> **Antwort:** *TODO*

### 3b) Abfall von $|\hat a_k|$ und Glattheit

In [ ]:
# Plot: |â_k| gegen k (log-log) für alle Signale, Steigungen bestimmen
# TODO

> **Frage:** Wie hängt der Abfall von $|\hat a_k|$ mit der Glattheit zusammen (Sprünge, Knicke)?
>
> **Antwort:** *TODO*

### 3c) Signal passt nicht ins Fenster

In [ ]:
# TODO: Sinus mit nicht ganzzahliger Periodenzahl im Fenster, Spektrum ansehen

In [ ]:
# TODO: Verbesserung, z. B. mit Fensterfunktion (Hann)

> **Frage:** Was passiert, wenn das Signal nicht ins Abtastfenster passt? Wie lässt sich das verbessern?
>
> **Antwort:** *TODO*

## 4 Zusammenfassung

*TODO*